In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import threading
import json
from dhanhq import dhanhq  as dh
from dhanhq import marketfeed  
import asyncio
import nest_asyncio
import pandas_ta as ta
import time
import concurrent.futures
import threading
import schedule
import logging
from dhanhq import marketfeed
import yfinance as yf
pd.set_option('display.max_columns', None)

DEBUG:matplotlib:matplotlib data path: /Users/Macbook/anaconda3/lib/python3.11/site-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/Users/Macbook/.matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is darwin
DEBUG:matplotlib:CACHEDIR=/Users/Macbook/.matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /Users/Macbook/.matplotlib/fontlist-v330.json


In [1]:
from NorenRestApiPy.NorenApi import  NorenApi
from threading import Timer
import pandas as pd
import time
import concurrent.futures
import logging

#enable dbug to see request and responses
logging.basicConfig(level=logging.DEBUG)


class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self
api=ShoonyaApiPy()        

In [3]:
user    = "FA194273"
pwd     = "Agr@271881"
factor2 = "752340"
vc      = "FA194273_U"
app_key = "24dec73fd5d525c6a2b1be21cafe213c"
imei    = "abc1234"
shoonya = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)


DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//QuickAuth
DEBUG:NorenRestApiPy.NorenApi:Req:jData={"source": "API", "apkversion": "1.0.0", "uid": "FA194273", "pwd": "a6b1af6fc77ac0dc039f9a356f4423fb264babea79927cb5ae3ce94f47ad2b91", "factor2": "752340", "vc": "FA194273_U", "appkey": "c31254bcd2d505abefc97d4cf0e953cb7921c2cade9888f144142c93cac58ff3", "imei": "abc1234"}
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443
DEBUG:urllib3.connectionpool:https://api.shoonya.com:443 "POST /NorenWClientTP//QuickAuth HTTP/1.1" 200 3408
DEBUG:NorenRestApiPy.NorenApi:Reply:{"request_time":"07:45:08 02-05-2025","actid":"FA194273","access_type":["WEB","TT","MOB","API"],"uname":"ATHARVA  AGARWAL","prarr":[{"prd":"C","s_prdt_ali":"CNC","exch":["NSE","BSE","NIPO","BSTAR"]},{"prd":"M","s_prdt_ali":"NRML","exch":["NFO","BFO","CDS","BCD"]},{"prd":"I","s_prdt_ali":"MIS","exch":["NSE","BSE","NFO","BFO","CDS","BCD"]},{"prd":"H","s_prdt_ali":"CO","exch":["

In [6]:
equ=pd.read_csv('https://lapi.kotaksecurities.com/wso2-scripmaster/v1/prod/2024-10-14/transformed/nse_cm.csv') # There is the presence of all the stock available , from kotak neo 
stock=pd.read_csv('/Users/Macbook/Desktop/n500.csv')  # This is the nifty 500 stock download from nse site
nifty500=equ[equ['pSymbolName'].isin(stock['Symbol'])]
nifty500=nifty500[nifty500['pGroup']=='EQ'].reset_index(drop=True)
nifty500.drop(index=45, inplace=True)
nifty500
symbols = nifty500['pTrdSymbol'].tolist()

In [7]:
print (symbols)

['NYKAA-EQ', 'AARTIIND-EQ', 'SUNTV-EQ', 'GPIL-EQ', 'SONATSOFTW-EQ', 'RATNAMANI-EQ', 'JKLAKSHMI-EQ', 'POLICYBZR-EQ', 'ALLCARGO-EQ', 'EMAMILTD-EQ', 'GMRINFRA-EQ', 'TECHM-EQ', 'GRINDWELL-EQ', 'ACE-EQ', 'ABB-EQ', 'ACC-EQ', 'ADANIENT-EQ', 'AEGISLOG-EQ', 'HAPPSTMNDS-EQ', 'PAYTM-EQ', 'SAPPHIRE-EQ', 'JINDALSTEL-EQ', 'IRCTC-EQ', 'JMFINANCIL-EQ', 'ELECON-EQ', 'LATENTVIEW-EQ', 'FLUOROCHEM-EQ', 'NAUKRI-EQ', 'GESHIP-EQ', 'TORNTPOWER-EQ', 'SOBHA-EQ', 'BSOFT-EQ', 'HBLPOWER-EQ', 'TANLA-EQ', 'STARHEALTH-EQ', 'ANANDRATHI-EQ', 'MAPMYINDIA-EQ', 'HCLTECH-EQ', 'METROBRAND-EQ', 'MEDPLUS-EQ', 'NETWORK18-EQ', 'UNOMINDA-EQ', 'NSLNISP-EQ', 'TIMKEN-EQ', 'DATAPATTNS-EQ', 'REDINGTON-EQ', 'PFC-EQ', 'RAJESHEXPO-EQ', 'GLENMARK-EQ', 'FSL-EQ', 'INDIANB-EQ', 'HSCL-EQ', 'IDEA-EQ', 'PAGEIND-EQ', 'ASTRAL-EQ', 'BALAMINES-EQ', 'PHOENIXLTD-EQ', 'FORTIS-EQ', 'NAVINFLUOR-EQ', 'INOXWIND-EQ', 'DLF-EQ', 'SPARC-EQ', 'ZYDUSLIFE-EQ', 'AVANTIFEED-EQ', 'CENTRALBK-EQ', 'KPRMILL-EQ', 'CIEINDIA-EQ', 'MOTILALOFS-EQ', 'CSBBANK-EQ', 'POWERGRI

In [9]:
import time
from datetime import datetime

def date_to_epoch(date_str):
    # Input format: "DD-MM-YYYY"
    dt = datetime.strptime(date_str, "%d-%m-%Y")
    return str(int(time.mktime(dt.timetuple())))

In [11]:
print(date_to_epoch("01-04-2025"))
print(date_to_epoch("01-05-2025"))

1743444900
1746036900


In [13]:
for symbol in symbols:
    try:
        # Get historical data (at least 15 candles)
        candles = api.get_daily_price_series(exchange='NSE', tradingsymbol=symbol, startdate="1743444900", enddate="1743444900")
        df = pd.DataFrame(candles)
        df.columns = ['datetime', 'open', 'high', 'low', 'close', 'volume']
        df['close'] = pd.to_numeric(df['close'], errors='coerce')
        
        # Calculate RSI-14
        df['rsi14'] = ta.rsi(df['close'], length=14)
        
        # Add stock symbol
        df['symbol'] = symbol

        # Append last row
        results.append(df.iloc[-1])
    
    except Exception as e:
        print(f"Error processing {symbol}: {e}")


DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//EODChartData
DEBUG:NorenRestApiPy.NorenApi:jData={"uid": "FA194273", "sym": "NSE:NYKAA-EQ", "from": "1743444900", "to": "1743444900"}&jKey=879aca5d0f79749f85d0e63436467c4ff4dbec8ff8d8ffc2e1826e61baea97be
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443
DEBUG:urllib3.connectionpool:https://api.shoonya.com:443 "POST /NorenWClientTP//EODChartData HTTP/1.1" 200 2
DEBUG:NorenRestApiPy.NorenApi:<Response [200]>
DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//EODChartData
DEBUG:NorenRestApiPy.NorenApi:jData={"uid": "FA194273", "sym": "NSE:AARTIIND-EQ", "from": "1743444900", "to": "1743444900"}&jKey=879aca5d0f79749f85d0e63436467c4ff4dbec8ff8d8ffc2e1826e61baea97be
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443


Error processing NYKAA-EQ: Length mismatch: Expected axis has 0 elements, new values have 6 elements


KeyboardInterrupt: 

In [41]:
ret =api.get_daily_price_series(exchange="NSE",tradingsymbol="AAVAS-EQ",startdate="1743444900",enddate="1746036900")

DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//EODChartData
DEBUG:NorenRestApiPy.NorenApi:jData={"uid": "FA194273", "sym": "NSE:AAVAS-EQ", "from": "1743444900", "to": "1746036900"}&jKey=879aca5d0f79749f85d0e63436467c4ff4dbec8ff8d8ffc2e1826e61baea97be
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443
DEBUG:urllib3.connectionpool:https://api.shoonya.com:443 "POST /NorenWClientTP//EODChartData HTTP/1.1" 200 3174
DEBUG:NorenRestApiPy.NorenApi:<Response [200]>


In [42]:
parsed_data = [json.loads(row) for row in ret]

In [43]:
take=pd.DataFrame(parsed_data)

In [44]:
take

,time,into,inth,intl,intc,ssboe,intv
0,30-APR-2025,1991.20,1996.60,1922.10,1942.90,1745971200,195275.00
1,29-APR-2025,1997.10,2087.00,1986.00,1991.20,1745884800,274564.00
2,28-APR-2025,2047.90,2060.20,1981.60,1991.50,1745798400,145769.00
3,25-APR-2025,2110.00,2150.00,1984.00,2038.10,1745539200,458897.00
4,24-APR-2025,2108.00,2168.00,2087.30,2096.20,1745452800,169622.00
5,23-APR-2025,2220.50,2224.10,2080.00,2119.40,1745366400,599926.00
6,22-APR-2025,2166.40,2234.00,2147.60,2220.50,1745280000,282678.00
7,21-APR-2025,2024.30,2199.00,2020.70,2166.30,1745193600,817238.00
8,17-APR-2025,2030.00,2034.00,1988.10,2024.30,1744848000,152637.00
9,16-APR-2025,2003.30,2040.00,1988.70,2030.90,1744761600,138389.00


In [48]:


# ----------------------------
# Helper: Convert DD-MM-YYYY to epoch
def date_to_epoch(date_str):
    dt = datetime.strptime(date_str, "%d-%m-%Y")
    return str(int(time.mktime(dt.timetuple())))

# ----------------------------
# Setup: Dates and API login
start_date = "01-04-2020"
end_date = "30-04-2025"

start_epoch = date_to_epoch(start_date)
end_epoch = date_to_epoch(end_date)

results = []

for sym in symbols:
    try:
        # Shoonya requires full sym format like 'NSE:PAYTM-EQ'
        full_sym = f"NSE:{sym}"

        # Fetch historical EOD data
        raw_data = api.get_daily_price_series(exchange="NSE", tradingsymbol=sym, startdate=start_epoch, enddate=end_epoch)

        if not raw_data or len(raw_data) < 15:
            print(f"Not enough data for {sym}")
            continue

        # Parse raw JSON strings
        parsed_data = [json.loads(row) for row in raw_data]

        # Build DataFrame
        df = pd.DataFrame(parsed_data)
        df.rename(columns={
            'time': 'date',
            'into': 'open',
            'inth': 'high',
            'intl': 'low',
            'intc': 'close',
            'intv': 'volume'
        }, inplace=True)

        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        df = df.sort_values('date')

        # Calculate RSI-14
        df['rsi14'] = ta.rsi(df['close'], length=14)

        # Add symbol info
        df['symbol'] = sym

        # Append the latest row
        results.append(df.iloc[-1])

    except Exception as e:
        print(f"Error processing {sym}: {e}")

# ----------------------------
# Final DataFrame with latest RSI per stock
final_df = pd.DataFrame(results)
final_df = final_df[['symbol', 'date', 'close', 'rsi14']]

# Show or save
print(final_df)
# final_df.to_csv("nifty500_rsi14_latest.csv", index=False)


DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//EODChartData
DEBUG:NorenRestApiPy.NorenApi:jData={"uid": "FA194273", "sym": "NSE:NYKAA-EQ", "from": "1585678500", "to": "1745950500"}&jKey=879aca5d0f79749f85d0e63436467c4ff4dbec8ff8d8ffc2e1826e61baea97be
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443
DEBUG:urllib3.connectionpool:https://api.shoonya.com:443 "POST /NorenWClientTP//EODChartData HTTP/1.1" 200 139374
DEBUG:NorenRestApiPy.NorenApi:<Response [200]>
DEBUG:NorenRestApiPy.NorenApi:https://api.shoonya.com/NorenWClientTP//EODChartData
DEBUG:NorenRestApiPy.NorenApi:jData={"uid": "FA194273", "sym": "NSE:AARTIIND-EQ", "from": "1585678500", "to": "1745950500"}&jKey=879aca5d0f79749f85d0e63436467c4ff4dbec8ff8d8ffc2e1826e61baea97be
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.shoonya.com:443
DEBUG:urllib3.connectionpool:https://api.shoonya.com:443 "POST /NorenWClientTP//EODChartData HTTP/1.1" 200 203698
DEBUG:

KeyboardInterrupt: 

In [46]:
final_df = pd.DataFrame(results)

In [47]:
final_df

,date,open,high,low,close,ssboe,volume,rsi14,symbol
0,29-APR-2025,193.90,199.74,193.50,194.72,1745884800,7143305.0,65.988391,NYKAA-EQ
0,29-APR-2025,433.90,438.35,428.00,430.80,1745884800,790637.0,60.171618,AARTIIND-EQ
0,29-APR-2025,647.00,652.35,640.20,644.30,1745884800,91327.0,51.472176,SUNTV-EQ
0,29-APR-2025,364.60,379.00,364.60,376.00,1745884800,8524645.0,57.826978,SONATSOFTW-EQ
0,29-APR-2025,2712.00,2725.00,2663.00,2704.80,1745884800,7659.0,56.248073,RATNAMANI-EQ
0,29-APR-2025,800.00,807.00,791.05,800.50,1745884800,49517.0,50.931442,JKLAKSHMI-EQ
0,29-APR-2025,1624.00,1636.80,1596.00,1600.60,1745884800,1052830.0,55.436076,POLICYBZR-EQ
0,29-APR-2025,629.95,639.80,626.00,635.30,1745884800,504931.0,66.442804,EMAMILTD-EQ
